# Lab — Aprendendo com dados não rotulados

Rotular dados é caro. Nesta aula vemos três formas de aproveitar dados **sem rótulo**, seguindo os slides:

1. **Deep Clustering (Deep k-Means):** o autoencoder aprende uma representação e, ao mesmo tempo, a organiza em clusters. Vocês escrevem a **loss de clustering**.
2. **Self-supervised / contrastive learning (SimCLR):** a supervisão vem dos próprios dados: duas augmentations da mesma imagem devem ficar próximas. Vocês escrevem a **loss NT-Xent**.
3. **Semi-supervised (FixMatch):** só 100 imagens rotuladas e 60 mil sem rótulo. Vocês escrevem a **loss não supervisionada** (pseudo-rótulo + consistência).

O SwAV, que também está nos slides, tem um notebook próprio (`SwAV.ipynb`).

**Como usar:** rode as células em ordem. Os trechos que vocês devem completar estão marcados com `# TODO`, e logo depois há uma célula de teste. Use GPU no Colab (*Ambiente de execução → Alterar tipo*). Cada treino leva de 1 a 2 minutos.

## 0. Setup

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
from torchvision.utils import make_grid
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# O MNIST inteiro cabe na GPU: dispensamos o DataLoader e fazemos as augmentations direto nos tensores
train_raw = datasets.MNIST("data", train=True, download=True)
test_raw = datasets.MNIST("data", train=False, download=True)
X_train = (train_raw.data.float() / 255).unsqueeze(1).to(device)  # (60000, 1, 28, 28), pixels em [0, 1]
y_train = train_raw.targets.to(device)
X_test = (test_raw.data.float() / 255).unsqueeze(1).to(device)    # (10000, 1, 28, 28)
y_test = test_raw.targets.to(device)
print("treino:", tuple(X_train.shape), "| teste:", tuple(X_test.shape))


def minibatches(n, batch_size, shuffle=True):
    idx = torch.randperm(n, device=device) if shuffle else torch.arange(n, device=device)
    for i in range(0, n, batch_size):
        yield idx[i:i + batch_size]


@torch.no_grad()
def extract(fn, X, batch_size=2048):
    """Aplica fn em X por partes (sem gradiente) e concatena o resultado."""
    return torch.cat([fn(X[i:i + batch_size]) for i in range(0, len(X), batch_size)])


def show_grid(imgs, nrow=10, title=None):
    g = make_grid(imgs.detach().cpu(), nrow=nrow, padding=2, pad_value=1.0)
    w = min(1.0 * nrow, 14)
    plt.figure(figsize=(w, w * g.shape[1] / g.shape[2]))
    plt.imshow(g.permute(1, 2, 0))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


def labeled_indices(n_per_class, seed=0):
    """Escolhe n_per_class imagens de cada dígito: o nosso pequeno conjunto rotulado."""
    g = torch.Generator().manual_seed(seed)
    y = y_train.cpu()
    idx = [torch.nonzero(y == c).flatten() for c in range(10)]
    idx = [i[torch.randperm(len(i), generator=g)[:n_per_class]] for i in idx]
    return torch.cat(idx).to(device)


# 3.000 imagens de teste fixas para as visualizações (t-SNE em 10 mil pontos demora)
VIS_IDX = torch.randperm(len(X_test), generator=torch.Generator().manual_seed(0))[:3000].to(device)


def plot_tsne(feats, title=None):
    """feats: {nome: características (len(VIS_IDX), d)}. Um t-SNE 2D por painel, colorido pelo dígito."""
    y = y_test[VIS_IDX].cpu().numpy()
    fig, axes = plt.subplots(1, len(feats), figsize=(4.5 * len(feats), 4.2), squeeze=False)
    for ax, (name, f) in zip(axes[0], feats.items()):
        emb = TSNE(n_components=2, init="pca", random_state=0).fit_transform(f.cpu().numpy())
        sc = ax.scatter(emb[:, 0], emb[:, 1], c=y, cmap="tab10", s=3, alpha=0.7, vmin=-0.5, vmax=9.5)
        ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(sc, ax=axes[0].tolist(), ticks=range(10), label="dígito (só para colorir)")
    if title:
        fig.suptitle(title)
    plt.show()

### A rede usada nas três partes

O `Backbone` transforma uma imagem `1×28×28` num vetor de **256 características**. Cada parte coloca uma "cabeça" diferente em cima dele:

| Parte | Cabeça | Treinada com |
|:---|:---|:---|
| 1. Deep k-Means | `Linear(256 → 10)` + decoder | reconstrução + clustering, **sem rótulos** |
| 2. SimCLR | projection head MLP `256 → 256 → 64` | loss contrastiva, **sem rótulos** |
| 3. FixMatch | `Linear(256 → 10)` (classificador) | 100 rótulos + 60 mil imagens sem rótulo |

Como o backbone é o mesmo, no fim da Parte 2 dá para comparar diretamente as representações aprendidas.

In [ ]:
class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 4, stride=2, padding=1), nn.ReLU(),   # 32x14x14
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(),  # 64x7x7
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256), nn.ReLU(),                 # 256 características
        )

    def forward(self, x):
        return self.net(x)


class Decoder(nn.Module):
    """O decoder do lab de Autoencoders: latent_dim -> 1x28x28."""
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 64 * 7 * 7), nn.ReLU(),
        )
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.ReLU(),  # 32x14x14
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1), nn.Sigmoid(),  # 1x28x28
        )

    def forward(self, z):
        return self.deconv(self.fc(z).view(-1, 64, 7, 7))

## 1. Deep Clustering: Deep k-Means

A ideia mais simples é **autoencoder + k-means**: treinar o AE, depois rodar k-means no espaço latente. O problema é que o AE aprende o latente só para reconstruir, sem nenhuma preocupação em formar grupos.

O **Deep k-Means** (Fard et al., 2020) treina as duas coisas juntas:

$$\mathcal{L}(x) = \underbrace{\mathcal{L}_{rec}(x)}_{\text{reconstrução}} + \lambda \underbrace{\sum_{k=1}^{K} \lVert h(x) - r_k \rVert^2 \, G_k\big(h(x), \alpha\big)}_{\mathcal{L}_{clust}(x)}$$

onde $h(x)$ é a representação de $x$, $r_k$ é o centróide do cluster $k$ (também aprendido) e $G_k$ é a pertinência **suave** de $x$ ao cluster $k$, uma softmax com temperatura $\alpha$:

$$G_k\big(h(x), \alpha\big) = \frac{e^{-\alpha \lVert h(x) - r_k \rVert^2}}{\sum_{k'} e^{-\alpha \lVert h(x) - r_{k'} \rVert^2}}$$

Nos slides: com $\alpha = 0$ todos os clusters pesam igual; com $\alpha$ grande, $G_k$ vira 0 ou 1 e recuperamos o k-means "duro". Durante o treino, **$\alpha$ aumenta**.

Um detalhe nosso: colocamos $h(x)$ e os centróides na **esfera unitária**. Sem isso, o encoder pode "trapacear" encolhendo todo o espaço latente, o que zera as distâncias sem formar clusters de verdade.

### 1.1 Pré-treino do autoencoder

In [ ]:
LATENT = 10


class AE(nn.Module):
    def __init__(self, latent_dim=LATENT):
        super().__init__()
        self.backbone = Backbone()
        self.to_latent = nn.Linear(256, latent_dim)
        self.decoder = Decoder(latent_dim)

    def embed(self, x):
        # h(x) na esfera unitária
        return F.normalize(self.to_latent(self.backbone(x)), dim=1)

    def forward(self, x):
        h = self.embed(x)
        return self.decoder(h), h


def recon_loss(x_hat, x):
    return F.binary_cross_entropy(x_hat, x, reduction="sum") / x.size(0)


def pretrain_ae(model, epochs=8, lr=1e-3, batch_size=256):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        total, n = 0.0, 0
        for idx in minibatches(len(X_train), batch_size):
            x = X_train[idx]
            x_hat, _ = model(x)
            loss = recon_loss(x_hat, x)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); n += 1
        print(f"época {ep + 1}: rec = {total / n:.2f}")
    return model


ae = pretrain_ae(AE())

### 1.2 Linha de base: k-means nos pixels e no latente do AE

Para medir a qualidade de um clustering usamos os rótulos **só na avaliação**. Como o cluster 3 não precisa corresponder ao dígito 3, a *clustering accuracy* primeiro encontra o melhor casamento entre clusters e classes (algoritmo húngaro) e depois calcula a acurácia, como nos slides.

In [ ]:
def cluster_acc(y_true, y_pred, k=10):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    counts = np.zeros((k, k), dtype=np.int64)
    np.add.at(counts, (y_pred, y_true), 1)               # counts[cluster, classe]
    rows, cols = linear_sum_assignment(-counts)          # casamento que maximiza os acertos
    return counts[rows, cols].sum() / len(y_true)


y_test_np = y_test.cpu().numpy()
sub = torch.randperm(len(X_train), device=device)[:10000]  # k-means em 10 mil imagens para ser rápido

# (a) k-means direto nos pixels
km_pix = KMeans(10, n_init=10, random_state=0).fit(X_train[sub].flatten(1).cpu().numpy())
acc_pix = cluster_acc(y_test_np, km_pix.predict(X_test.flatten(1).cpu().numpy()))

# (b) k-means no espaço latente do AE pré-treinado
H_sub = extract(ae.embed, X_train[sub]).cpu().numpy()
km_ae = KMeans(10, n_init=10, random_state=0).fit(H_sub)
acc_ae = cluster_acc(y_test_np, km_ae.predict(extract(ae.embed, X_test).cpu().numpy()))

print(f"k-means nos pixels:        ACC = {acc_pix:.3f}")
print(f"AE + k-means (latente):    ACC = {acc_ae:.3f}")

# Guardamos o latente de agora para comparar com o depois do Deep k-Means
H_vis_before = extract(ae.embed, X_test[VIS_IDX])

### TODO 1 — a loss de clustering do Deep k-Means

Dado um batch de representações `h` (`B × d`) e os centróides `centroids` (`K × d`), calcule

$$\mathcal{L}_{clust} = \frac{1}{B}\sum_{i=1}^{B} \sum_{k=1}^{K} \lVert h_i - r_k \rVert^2 \, G_k(h_i, \alpha)$$

Dicas:
- As distâncias ao quadrado de todos contra todos saem com *broadcasting*: `h[:, None, :] - centroids[None, :, :]` tem forma `B × K × d`.
- $G$ é uma softmax sobre os clusters de `-alpha * distâncias`.

In [ ]:
def dkm_clustering_loss(h, centroids, alpha):
    # TODO: distâncias ao quadrado (B, K) entre cada h_i e cada centróide r_k
    # TODO: G = softmax de -alpha * distâncias, sobre os clusters
    # TODO: soma em k de G * distância, média no batch
    raise NotImplementedError

In [ ]:
# Teste do TODO 1
h = F.normalize(torch.randn(6, 3), dim=1)
r = F.normalize(torch.randn(4, 3), dim=1)
d2 = torch.cdist(h, r) ** 2
assert torch.allclose(dkm_clustering_loss(h, r, 0.0), d2.mean(), atol=1e-5), \
    "com alpha = 0 todos os clusters pesam 1/K: a loss é a média das distâncias"
assert torch.allclose(dkm_clustering_loss(h, r, 1e4), d2.min(dim=1).values.mean(), atol=1e-4), \
    "com alpha grande a loss vira a do k-means: distância ao centróide mais próximo"
print("dkm_clustering_loss ok!")

### 1.3 Treino conjunto

Os centróides começam no resultado do k-means da linha de base (b) e passam a ser parâmetros treináveis. A cada época $\alpha$ cresce de 10 até 1000.

In [ ]:
def train_dkm(model, init_centroids, alphas, lam=20.0, lr=1e-3, batch_size=256):
    centroids = nn.Parameter(torch.tensor(init_centroids, dtype=torch.float32, device=device))
    opt = torch.optim.Adam(list(model.parameters()) + [centroids], lr=lr)
    for ep, alpha in enumerate(alphas):
        tot_rec, tot_clu, n = 0.0, 0.0, 0
        for idx in minibatches(len(X_train), batch_size):
            x = X_train[idx]
            x_hat, h = model(x)
            rec = recon_loss(x_hat, x)
            clu = dkm_clustering_loss(h, F.normalize(centroids, dim=1), alpha)
            loss = rec + lam * clu
            opt.zero_grad(); loss.backward(); opt.step()
            tot_rec += rec.item(); tot_clu += clu.item(); n += 1
        pred = dkm_predict(model, centroids)
        print(f"época {ep + 1:2d} | alpha = {alpha:7.1f} | rec = {tot_rec / n:6.2f} | "
              f"clust = {tot_clu / n:.4f} | ACC teste = {cluster_acc(y_test_np, pred):.3f}")
    return F.normalize(centroids.detach(), dim=1)


@torch.no_grad()
def dkm_predict(model, centroids, X=X_test):
    H = extract(model.embed, X)
    return torch.cdist(H, F.normalize(centroids, dim=1)).argmin(dim=1).cpu().numpy()


alphas = np.geomspace(10, 1000, 10)
centroids = train_dkm(ae, km_ae.cluster_centers_, alphas)

In [ ]:
pred_dkm = dkm_predict(ae, centroids)
print(f"k-means nos pixels:        ACC = {acc_pix:.3f}")
print(f"AE + k-means (latente):    ACC = {acc_ae:.3f}")
print(f"Deep k-Means:              ACC = {cluster_acc(y_test_np, pred_dkm):.3f}")

# 10 imagens de teste mais próximas de cada centróide (uma linha por cluster)
H_test = extract(ae.embed, X_test)
nearest = torch.cdist(centroids, H_test).topk(10, dim=1, largest=False).indices
show_grid(X_test[nearest.flatten()], nrow=10, title="Cada linha: as 10 imagens mais próximas de um centróide")

### 1.4 Olhando o espaço latente

O latente tem 10 dimensões; para desenhar, usamos **t-SNE**, que projeta em 2D tentando manter perto o que estava perto. As cores são os dígitos verdadeiros, usados **só para pintar** os pontos. Compare o latente do AE antes e depois do treino conjunto.

Cuidado ao ler um t-SNE: ele preserva **vizinhanças**, não distâncias. O tamanho de um grupo e a distância entre dois grupos no gráfico não querem dizer muita coisa; o que importa é se cada cor forma um grupo próprio.

In [ ]:
plot_tsne({
    "AE pré-treinado": H_vis_before,
    "depois do Deep k-Means": extract(ae.embed, X_test[VIS_IDX]),
})

**Pergunte-se:**
- Algum cluster mistura dois dígitos? Quais? Faz sentido visualmente? Esses dígitos também ficam misturados no t-SNE?
- O que mudou no t-SNE depois do treino conjunto? Os grupos ficaram mais compactos?
- Rode de novo com `lam=0` (só reconstrução, mas com os centróides "andando"). O que acontece com a ACC? E com `lam=200`, a reconstrução piora?
- Rode de novo começando com $\alpha = 1$ (`np.geomspace(1, 1000, 10)`). O que acontece com a ACC nas primeiras épocas? Pense no que $G_k$ vale quando $\alpha \to 0$: para onde a loss puxa cada ponto?
- Em nenhum momento usamos rótulos para treinar. De onde vem a informação que separa os dígitos?

## 2. Self-supervised: SimCLR

> *"A form of unsupervised learning where the data provides the supervision"* (A. Zisserman)

O SimCLR (Chen et al., 2020) gera **duas visões** de cada imagem com augmentations fortes e treina a rede para que as duas visões fiquem próximas e longe das visões das outras imagens do batch:

`imagem → duas augmentations → backbone (h) → projection head (z) → loss contrastiva`

### 2.1 Augmentations

O SimCLR original usa *random crop*, distorção de cor e blur. Para dígitos em tons de cinza usamos o equivalente: rotação/escala/translação aleatórias (crop), mudança de intensidade (cor), *cutout* e ruído. Tudo roda em batch na GPU. As mesmas funções servem para o FixMatch na Parte 3.

In [ ]:
def random_affine(x, max_rot=0.0, max_scale=0.0, max_shift=0.0):
    """Rotação (graus), escala e translação (pixels) aleatórias, diferentes para cada imagem do batch."""
    B = x.size(0)

    def U(a):  # uniforme em [-a, a]
        return (torch.rand(B, device=x.device) * 2 - 1) * a

    ang = U(max_rot) * math.pi / 180
    s = 1 + U(max_scale)
    tx, ty = U(max_shift) * 2 / x.size(-1), U(max_shift) * 2 / x.size(-1)  # pixels -> coordenadas em [-1, 1]
    cos, sin = torch.cos(ang) / s, torch.sin(ang) / s
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, align_corners=False)


def cutout(x, size):
    """Apaga um quadrado size x size em posição aleatória de cada imagem."""
    B, _, H, W = x.shape
    cy = torch.randint(0, H, (B, 1, 1), device=x.device)
    cx = torch.randint(0, W, (B, 1, 1), device=x.device)
    yy = torch.arange(H, device=x.device).view(1, H, 1)
    xx = torch.arange(W, device=x.device).view(1, 1, W)
    mask = ((yy - cy).abs() <= size // 2) & ((xx - cx).abs() <= size // 2)
    return x * (~mask).unsqueeze(1)


def weak_aug(x):
    return random_affine(x, max_shift=2)


def strong_aug(x):
    x = random_affine(x, max_rot=25, max_scale=0.25, max_shift=4)
    x = x * (0.5 + 0.5 * torch.rand(x.size(0), 1, 1, 1, device=x.device))  # intensidade
    x = cutout(x, 10)
    return (x + 0.1 * torch.randn_like(x)).clamp(0, 1)


x8 = X_train[:10]
show_grid(torch.cat([x8, weak_aug(x8), strong_aug(x8), strong_aug(x8)]), nrow=10,
          title="Original / augmentation fraca / duas augmentations fortes")

### TODO 2 — a loss NT-Xent

Para um batch de $N$ imagens temos $2N$ visões. As projeções são normalizadas e comparadas por similaridade de cosseno. Para um par positivo $(i, j)$ (as duas visões da mesma imagem):

$$\ell_{i,j} = -\log \frac{\exp(\mathrm{sim}(z_i, z_j)/\tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\mathrm{sim}(z_i, z_k)/\tau)}$$

e a loss final é a média sobre todos os $2N$ pares $(i,j)$ e $(j,i)$. Os outros $2(N-1)$ exemplos do batch são os negativos.

Repare que $\ell_{i,j}$ é exatamente uma **cross-entropy**: os "logits" da linha $i$ são as similaridades com todos os outros exemplos, e a "classe correta" é a posição do positivo.

Dicas:
- Concatene `z1` e `z2` (`2N × d`) e normalize com `F.normalize`. A matriz de similaridades é `z @ z.T / tau`.
- Tire a diagonal (similaridade de cada um consigo mesmo) colocando `-inf` nela: `masked_fill`.
- Na linha `i < N` o positivo está na coluna `i + N`; na linha `i ≥ N`, na coluna `i - N`.

In [ ]:
def nt_xent(z1, z2, tau=0.5):
    N = z1.size(0)
    # TODO: concatenar z1 e z2 e normalizar (2N, d)
    # TODO: matriz de similaridades (2N, 2N) dividida por tau, com -inf na diagonal
    # TODO: índice do positivo de cada linha e cross-entropy
    raise NotImplementedError

In [ ]:
# Teste do TODO 2: compara com uma implementação "na mão", termo a termo
def nt_xent_loop(z1, z2, tau):
    z = F.normalize(torch.cat([z1, z2]), dim=1)
    N2 = len(z)
    total = 0.0
    for i in range(N2):
        j = (i + N2 // 2) % N2
        den = sum(torch.exp(z[i] @ z[k] / tau) for k in range(N2) if k != i)
        total += -torch.log(torch.exp(z[i] @ z[j] / tau) / den)
    return total / N2

torch.manual_seed(0)
a, b = torch.randn(5, 8), torch.randn(5, 8)
for tau in [0.1, 0.5, 1.0]:
    assert torch.allclose(nt_xent(a, b, tau), nt_xent_loop(a, b, tau), atol=1e-4), f"diferente da referência (tau={tau})"
assert nt_xent(a, a + 0.01 * torch.randn(5, 8)) < nt_xent(a, b), "visões parecidas deveriam ter loss menor"
print("nt_xent ok!")

### 2.2 Treino

In [ ]:
class SimCLR(nn.Module):
    def __init__(self, proj_dim=64):
        super().__init__()
        self.backbone = Backbone()                    # h: a representação que vamos usar depois
        self.head = nn.Sequential(                    # g: projection head, só para a loss
            nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, proj_dim)
        )

    def forward(self, x):
        return self.head(self.backbone(x))


def train_simclr(model, epochs=15, tau=0.5, lr=1e-3, batch_size=512):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        total, n = 0.0, 0
        for idx in minibatches(len(X_train), batch_size):
            x = X_train[idx]                          # repare: y_train nunca aparece aqui
            loss = nt_xent(model(strong_aug(x)), model(strong_aug(x)), tau)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); n += 1
        print(f"época {ep + 1:2d}: NT-Xent = {total / n:.3f}")
    return model


simclr = train_simclr(SimCLR())

### 2.3 Avaliação linear

Como saber se a representação é boa sem usar rótulos no treino? O protocolo padrão é a **avaliação linear**: congelamos o backbone e treinamos **só uma camada linear** em cima das 256 características. Se uma reta já separa bem as classes, a representação organizou os dados.

Comparamos três backbones: aleatório (sem treino), o do Deep k-Means da Parte 1 e o do SimCLR. E dois cenários: só **10 rótulos por classe** (100 imagens) ou os 60 mil rótulos.

In [ ]:
def linear_probe(backbone, n_per_class=None, epochs=300, lr=1e-2):
    backbone.eval()
    F_tr, F_te = extract(backbone, X_train), extract(backbone, X_test)
    mu, sd = F_tr.mean(0), F_tr.std(0) + 1e-6
    F_tr, F_te = (F_tr - mu) / sd, (F_te - mu) / sd
    idx = labeled_indices(n_per_class) if n_per_class else torch.arange(len(X_train), device=device)
    clf = nn.Linear(F_tr.size(1), 10).to(device)
    opt = torch.optim.Adam(clf.parameters(), lr=lr, weight_decay=1e-4)
    for _ in range(epochs):                           # batch completo: são só vetores de 256 números
        loss = F.cross_entropy(clf(F_tr[idx]), y_train[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (clf(F_te).argmax(1) == y_test).float().mean().item()


torch.manual_seed(0)
backbones = {
    "aleatório (sem treino)": Backbone().to(device),
    "Deep k-Means (Parte 1)": ae.backbone,
    "SimCLR": simclr.backbone,
}
print(f"{'backbone':<26}{'10 rótulos/classe':>20}{'todos os rótulos':>20}")
for name, bb in backbones.items():
    print(f"{name:<26}{linear_probe(bb, 10):>20.3f}{linear_probe(bb):>20.3f}")

### 2.4 Olhando as representações

As mesmas 256 características que a avaliação linear usou, projetadas em 2D com t-SNE. Nenhum dos três backbones viu um rótulo.

In [ ]:
plot_tsne({name: extract(bb, X_test[VIS_IDX]) for name, bb in backbones.items()})

**Pergunte-se:**
- Com só 100 rótulos, qual a diferença entre o backbone aleatório e o do SimCLR? E com todos os rótulos, a diferença diminui? Por quê?
- O t-SNE do backbone aleatório já mostra alguns grupos. Por que uma rede **sem treino** já separa um pouco os dígitos? (Pense no k-means nos pixels da Parte 1.)
- Compare os t-SNEs com a tabela: o backbone com grupos mais separados é o que tem a melhor avaliação linear?
- Os slides dizem que a representação **antes** da projection head ($h$) é melhor que a de depois ($z$). Teste: troque `simclr.backbone` por `nn.Sequential(simclr.backbone, simclr.head)` (ajuste a dimensão do `nn.Linear` se precisar).
- Treine de novo com `tau=0.1` e `tau=1.0`. Como a temperatura muda o resultado?
- Tire uma das augmentations de `strong_aug` (por exemplo, o `random_affine`). O que acontece? Qual augmentation parece mais importante?

## 3. Semi-supervised: FixMatch

Agora o cenário realista: temos **100 imagens rotuladas** (10 por dígito) e as 60 mil imagens sem rótulo. O FixMatch (Sohn et al., 2020) combina duas ideias:

- **Pseudo-labeling:** o próprio modelo rotula as imagens sem rótulo.
- **Consistência:** a previsão para uma versão **fortemente** aumentada deve bater com a previsão para uma versão **fracamente** aumentada da mesma imagem.

A cada passo:

1. **Loss supervisionada** $\ell_s$: cross-entropy no batch rotulado (augmentation fraca).
2. Para o batch sem rótulo ($\mu$ vezes maior), calcula a previsão na **visão fraca**: $q_b = p(y \mid \alpha(u_b))$.
3. **Loss não supervisionada** $\ell_u$: se o modelo está confiante ($\max q_b \ge \tau$), o pseudo-rótulo $\hat{q}_b = \arg\max q_b$ vira alvo da **visão forte**:

$$\ell_u = \frac{1}{\mu B}\sum_{b=1}^{\mu B} \mathbb{1}\big(\max q_b \ge \tau\big)\; \mathrm{H}\big(\hat{q}_b,\ p(y \mid \mathcal{A}(u_b))\big)$$

4. Loss total: $\ell_s + \lambda_u \ell_u$.

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = Backbone()
        self.fc = nn.Linear(256, 10)

    def forward(self, x):
        return self.fc(self.backbone(x))


lab_idx = labeled_indices(10)          # 100 imagens rotuladas
X_lab, y_lab = X_train[lab_idx], y_train[lab_idx]
show_grid(X_lab[::10], nrow=10, title="Uma imagem rotulada de cada classe (temos 10 de cada)")


@torch.no_grad()
def test_accuracy(model):
    model.eval()
    acc = (extract(model, X_test).argmax(1) == y_test).float().mean().item()
    model.train()
    return acc

### TODO 3 — a loss não supervisionada do FixMatch

Recebe os logits da visão fraca e da visão forte das **mesmas** imagens sem rótulo e devolve $\ell_u$ e a fração de exemplos que passaram no limiar (para acompanharmos o treino).

Dicas:
- O pseudo-rótulo é um **alvo fixo**, como um rótulo de verdade: use `.detach()` nos logits fracos para deixar isso explícito.
- `probs.max(dim=1)` devolve a confiança e a classe ao mesmo tempo.
- Calcule a cross-entropy por exemplo (`reduction="none"`), multiplique pela máscara e tire a média sobre **todos** os exemplos (inclusive os mascarados), como na fórmula.

In [ ]:
def fixmatch_unsup_loss(logits_weak, logits_strong, tau=0.95):
    # TODO: probabilidades da visão fraca, sem gradiente
    # TODO: confiança e pseudo-rótulo de cada exemplo; máscara = confiança >= tau
    # TODO: cross-entropy da visão forte contra o pseudo-rótulo, só onde a máscara vale 1
    # devolva (loss, fração de exemplos com máscara = 1)
    raise NotImplementedError

In [ ]:
# Teste do TODO 3
weak = torch.tensor([[10.0, 0.0, 0.0], [0.0, 0.1, 0.0]], requires_grad=True)  # 1º confiante, 2º não
strong = torch.tensor([[0.0, 2.0, 0.0], [5.0, 0.0, 0.0]], requires_grad=True)
loss, rate = fixmatch_unsup_loss(weak, strong, tau=0.95)
expected = F.cross_entropy(strong[:1], torch.tensor([0])) / 2   # só o 1º conta, média sobre os 2
assert abs(rate.item() - 0.5) < 1e-6, "metade dos exemplos passa no limiar"
assert torch.allclose(loss, expected, atol=1e-5), "loss diferente do esperado"
loss.backward()
assert weak.grad is None or weak.grad.abs().sum() == 0, "o gradiente não pode passar pelo pseudo-rótulo"
assert strong.grad[1].abs().sum() == 0, "exemplos abaixo do limiar não devem contribuir"
print("fixmatch_unsup_loss ok!")

### 3.1 Treino: só os 100 rótulos vs FixMatch

A mesma função treina os dois. Com `use_unlabeled=False` ela ignora as imagens sem rótulo e vira um treino supervisionado comum.

In [ ]:
def train_semi(model, steps=2000, batch_size=64, mu=7, lam_u=1.0, tau=0.95,
               use_unlabeled=True, lr=1e-3, eval_every=250):
    model.to(device).train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(1, steps + 1):
        li = torch.randint(0, len(X_lab), (batch_size,), device=device)
        loss_s = F.cross_entropy(model(weak_aug(X_lab[li])), y_lab[li])
        loss_u, mask_rate = torch.zeros((), device=device), torch.zeros((), device=device)
        if use_unlabeled:
            xu = X_train[torch.randint(0, len(X_train), (mu * batch_size,), device=device)]  # sem rótulo
            logits_w, logits_s = model(torch.cat([weak_aug(xu), strong_aug(xu)])).chunk(2)
            loss_u, mask_rate = fixmatch_unsup_loss(logits_w, logits_s, tau)
        loss = loss_s + lam_u * loss_u
        opt.zero_grad(); loss.backward(); opt.step()
        if step % eval_every == 0:
            print(f"passo {step:5d} | l_s = {loss_s.item():.3f} | l_u = {loss_u.item():.3f} | "
                  f"máscara = {mask_rate.item():.2f} | acc teste = {test_accuracy(model):.3f}")
    return model


torch.manual_seed(0)
print("== Só os 100 rótulos ==")
sup_only = train_semi(Classifier(), use_unlabeled=False)
torch.manual_seed(0)
print("\n== FixMatch: 100 rótulos + 60 mil sem rótulo ==")
fixmatch = train_semi(Classifier())

In [ ]:
print(f"Só os 100 rótulos:   acc teste = {test_accuracy(sup_only):.3f}")
print(f"FixMatch:            acc teste = {test_accuracy(fixmatch):.3f}")

**Pergunte-se:**
- Acompanhe a coluna `máscara`. Por que ela começa baixa e sobe? O que aconteceria se o modelo começasse confiante e **errado**?
- Treine com `tau=0.5` e com `tau=0.99`. Qual é o compromisso entre quantidade e qualidade dos pseudo-rótulos?
- Troque `strong_aug` por `weak_aug` na visão forte (dentro de `train_semi`). A consistência ainda ajuda? Por quê?
- Junte as Partes 2 e 3: comece o FixMatch com o backbone do SimCLR (`model.backbone.load_state_dict(simclr.backbone.state_dict())`). Melhora? Converge mais rápido?

# Para ir além

Escolha um experimento, explique o que fez e compartilhe seus resultados no [issue do GitHub](https://github.com/Erickslb/deep-learning-fgv-2026/issues), que todo mundo vê.

1. **Rótulos ruidosos:** corrompa 40% de `y_lab` (troque por classes aleatórias) e treine o FixMatch. O pseudo-labeling ajuda ou piora? Como você detectaria os rótulos errados? (Veja o SELFIE nos slides: amostras com loss baixa tendem a estar corretas.)
2. **Menos rótulos:** repita a Parte 3 com 1, 2 e 5 rótulos por classe. A partir de quando o FixMatch para de funcionar?
3. **Deep Clustering:** troque o k-means duro da inicialização por centróides aleatórios. O Deep k-Means ainda chega lá? E o que acontece com $\alpha$ fixo em 1000 desde o início?
4. **SimCLR + clustering:** rode k-means nas características do SimCLR e calcule a `cluster_acc`. Compare com o Deep k-Means.
5. **FashionMNIST:** troque `datasets.MNIST` por `datasets.FashionMNIST` e repita tudo. Qual método sofre mais?